# 05. Secure AI Development

## 📚 Learning Objectives

By completing this notebook, you will:
- Apply secure development practices to AI
- Identify and mitigate security threats
- Harden ML pipelines and deployments

## 🔗 Prerequisites

- ✅ Basic Python
- ✅ Basic NumPy/Pandas (when applicable)

---

---

# 05. Secure AI Development

## 🚨 THE PROBLEM: We Need Secure AI Systems

**Remember the limitation from the previous notebook?**

We learned GDPR compliance requirements and practices. But we discovered:

**How do we build secure AI systems that protect against attacks and vulnerabilities?**

**The Problem**: Secure AI systems also need:
- ❌ **Security measures** against attacks (adversarial, data poisoning)
- ❌ **Vulnerability management** (identify and fix security issues)
- ❌ **Secure coding practices** (prevent security bugs)
- ❌ **Security testing** (penetration testing, security audits)

**We've learned:**
- ✅ How to use basic data protection (Notebook 1)
- ✅ How to use advanced privacy technologies (Notebook 2)
- ✅ How to use differential privacy (Notebook 3)
- ✅ How to ensure GDPR compliance (Notebook 4)
- ✅ Privacy and compliance practices

**But we haven't learned:**
- ❌ How to **protect against adversarial attacks**
- ❌ How to **manage security vulnerabilities**
- ❌ How to **implement secure coding practices**
- ❌ How to **test for security issues**

**We need secure development practices** to:
1. Protect against adversarial attacks
2. Manage security vulnerabilities
3. Implement secure coding practices
4. Test for security issues

**This notebook solves that problem** by teaching you secure AI development practices!

---

## 📚 Prerequisites (What You Need First)

**BEFORE starting this notebook**, you should have completed:
- ✅ **Example 1: Data Protection** - Understanding basic protection
- ✅ **Example 2: Privacy Technologies** - Understanding PETs
- ✅ **Example 3: Differential Privacy** - Understanding privacy guarantees
- ✅ **Example 4: GDPR Compliance** - Understanding regulatory compliance
- ✅ **Basic Python knowledge**: Functions, data manipulation

**If you haven't completed these**, you might struggle with:
- Understanding why security matters for AI
- Knowing common security vulnerabilities
- Understanding secure coding practices

---

## 🔗 Where This Notebook Fits

**This is the FIFTH core example in Unit 3** - it teaches you secure development! (Two hands-on practice notebooks, 06 and 07, follow before you move on to Unit 4.)

**Why this example LAST?**
- **Before** you can secure systems, you need privacy techniques (Examples 1-3)
- **Before** you can secure systems, you need compliance (Example 4)
- **Before** you can deploy systems, you need security

**Builds on**: 
- 📓 Example 1: Data Protection (basic protection strategies)
- 📓 Example 2: Privacy Technologies (advanced PETs)
- 📓 Example 3: Differential Privacy (privacy guarantees)
- 📓 Example 4: GDPR Compliance (regulatory compliance)

**Leads to**: 
- 📓 Unit 4: Transparency and Accountability (next unit in the course!)

**Why this order?**
1. Secure development provides **security practices** (needed for safe deployment)
2. Secure development teaches **vulnerability management** (critical for protection)
3. Secure development shows **complete security workflow** (development to deployment)

---

## The Story: Building Secure Systems

Imagine you're building a house. **Before** you finish, you need security - locks, alarms, fire safety. **After** implementing security, you have a safe, protected house!

Same with AI: **Before** we have privacy and compliance but may not be secure, now we learn secure development - protect against attacks, manage vulnerabilities, implement secure coding! **After** secure development, we have secure, private, and compliant AI systems!

---

## Why Secure Development Matters

Secure development is essential for ethical AI:
- **Protection**: Protect against adversarial attacks and vulnerabilities
- **Trust**: Build user confidence in secure systems
- **Compliance**: Meet security requirements
- **Risk Mitigation**: Prevent security breaches and data exposure
- **Best Practices**: Follow industry security standards

## Learning Objectives
1. Understand security vulnerabilities in AI systems
2. Learn secure coding practices
3. Understand adversarial attacks and defenses
4. Implement security testing
5. Create security incident response plans
6. Understand secure deployment practices

## 📥 Inputs & 📤 Outputs

**Inputs:** What we use in this notebook

- `../../../Course 04/datasets/raw/cicids2017.csv` - the **CICIDS2017** intrusion
  detection dataset: real captured network flows labelled `BENIGN` or with the real
  attack that produced them (`FTP-Patator`, `SSH-Patator` - brute-force attacks).
  We attack a model trained on real attack traffic, which is the only way to see
  what these threats actually do.
- The file is 707 MB, so we read a classroom-size slice with `usecols` and `nrows`
  and say exactly what we sampled.

**Outputs:** What you'll see when you run the cells

- A near-perfect intrusion detector, and how little perturbation it takes to blind it
- The damage that poisoned training labels do at four contamination levels
- A cryptographic fingerprint that catches a single altered label

> **The randomness here is the attack, not the data.** The perturbations and label
> flips are the threat being demonstrated, deliberately injected into a real dataset.

---

## Part 1: Three Security Threats to AI Systems - Demonstrated

We demonstrate three classic AI-security concerns on a **real intrusion-detection
model** trained on **real attack traffic**:

1. **Adversarial perturbations**: small input changes that let an attack slip past
2. **Data poisoning**: corrupted training labels that damage the model
3. **Integrity checking**: detecting that training data was tampered with

The threat model matters. A real attacker does not add noise to *everything* - they
modify **their own traffic** to look benign while still doing the attack. So we
perturb only the attack flows and measure what the detector still catches.

In [1]:
# Why: models that ace clean test sets can crumble under small input changes -
# measuring that fragility BEFORE deployment is basic security hygiene.

# Step 1: Adversarial-perturbation robustness test on a REAL intrusion detector

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, recall_score

print("="*80)
print("⚔️  ADVERSARIAL PERTURBATION TEST (real network traffic)")
print("="*80)

# Load a slice of CICIDS2017: real captured network flows with real attack labels.
# The full file is 707 MB / 79 columns, so we take 9 flow features and the label
# from the first 400,000 rows - enough attack traffic to train on, fast to read.
FLOW_COLS = ['Destination Port', 'Flow Duration', 'Total Fwd Packets',
             'Total Backward Packets', 'Flow Bytes/s', 'Flow Packets/s',
             'Fwd Packet Length Mean', 'Bwd Packet Length Mean',
             'Flow IAT Mean', 'Label']
flows = pd.read_csv('../../../Course 04/datasets/raw/cicids2017.csv',
                    usecols=lambda c: c.strip() in FLOW_COLS, nrows=400_000)
flows.columns = [c.strip() for c in flows.columns]

# Real capture data is messy: division-by-zero in the rate columns produces real
# infinities and NaNs. Drop them and say how many - silent cleaning hides problems.
before = len(flows)
flows = flows.replace([np.inf, -np.inf], np.nan).dropna()
print(f"\nRead {before:,} real network flows; dropped "
      f"{before - len(flows):,} rows with infinite/missing rate values "
      f"(real artefacts of division by zero in the capture).")

print("\nReal traffic labels in this slice:")
print(flows['Label'].value_counts().to_string())

flows['attack'] = (flows['Label'] != 'BENIGN').astype(int)

# Classroom-size, class-balanced-ish sample so the notebook runs in seconds.
attacks = flows[flows['attack'] == 1].sample(6000, random_state=42)
benign = flows[flows['attack'] == 0].sample(20000, random_state=42)
sample = pd.concat([attacks, benign]).sample(frac=1, random_state=42)
FEATURES = [c for c in flows.columns if c not in ('Label', 'attack')]
print(f"\nWorking sample: {len(sample):,} flows "
      f"({int(sample['attack'].sum()):,} real attacks, {len(benign):,} benign), "
      f"random_state=42")

X = sample[FEATURES].values.astype(float)
y = sample['attack'].values
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)

# Train the baseline detector and record its clean-data numbers as the reference.
model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)
clean_pred = model.predict(X_test)
clean_acc = accuracy_score(y_test, clean_pred)
clean_recall = recall_score(y_test, clean_pred)
print(f"\nClean test accuracy      : {clean_acc:.4f}")
print(f"Clean attack DETECTION   : {clean_recall:.4f}  "
      f"(share of real attacks caught)")
print("On the clean test set this looks like a finished, production-ready detector.")

# --- The attack ---
# Threat model: the attacker controls their OWN traffic. They nudge the flow
# features of their attacks - padding packets, adding delays - while leaving
# benign traffic alone. Perturbation size is expressed in units of each
# feature's training standard deviation, so it is comparable across features.
feature_sd = X_train.std(axis=0)
rng = np.random.default_rng(0)

print("\nAttacker perturbs ONLY their own attack flows:")
print(f"{'Perturbation':>14}{'Attacks caught':>17}{'Change':>10}{'Overall acc':>14}")
evasion = []
for eps in [0.01, 0.05, 0.10, 0.25]:
    X_adv = X_test.copy()
    is_attack = y_test == 1
    X_adv[is_attack] = X_adv[is_attack] + rng.normal(
        0, eps * feature_sd, X_adv[is_attack].shape)
    adv_pred = model.predict(X_adv)
    r = recall_score(y_test, adv_pred)
    a = accuracy_score(y_test, adv_pred)
    evasion.append({'eps': eps, 'recall': r, 'accuracy': a})
    print(f"{eps:>13.0%}{r:>17.4f}{r - clean_recall:>+10.4f}{a:>14.4f}")

worst = evasion[0]
print(f"\n⚠️  A perturbation of just {worst['eps']:.0%} of one standard deviation")
print(f"   dropped attack detection from {clean_recall:.1%} to {worst['recall']:.1%}.")
print(f"   The detector that scored {clean_acc:.2%} on clean data now misses")
print(f"   roughly {1 - worst['recall']:.0%} of the attacks it was built to stop.")
print("\nNote the second trap: overall ACCURACY stayed high "
      f"({worst['accuracy']:.2%}) throughout,")
print("because the benign traffic - the large majority - was untouched. Accuracy")
print("is the wrong metric for a security model; detection rate is the right one.")
print("\nAnd this is a LOWER BOUND on the danger: real attacks (FGSM, PGD) choose")
print("the WORST-case direction rather than a random one, and do more damage at")
print("the same perturbation size.")

⚔️  ADVERSARIAL PERTURBATION TEST (real network traffic)



Read 400,000 real network flows; dropped 243 rows with infinite/missing rate values (real artefacts of division by zero in the capture).

Real traffic labels in this slice:
Label
BENIGN         386697
FTP-Patator      7935
SSH-Patator      5125

Working sample: 26,000 flows (6,000 real attacks, 20,000 benign), random_state=42

Clean test accuracy      : 0.9994
Clean attack DETECTION   : 0.9989  (share of real attacks caught)
On the clean test set this looks like a finished, production-ready detector.

Attacker perturbs ONLY their own attack flows:
  Perturbation   Attacks caught    Change   Overall acc
           1%           0.2328   -0.7661        0.8226
           5%           0.1361   -0.8628        0.8003


          10%           0.0744   -0.9244        0.7860
          25%           0.0233   -0.9756        0.7742

⚠️  A perturbation of just 1% of one standard deviation
   dropped attack detection from 99.9% to 23.3%.
   The detector that scored 99.94% on clean data now misses
   roughly 77% of the attacks it was built to stop.

Note the second trap: overall ACCURACY stayed high (82.26%) throughout,
because the benign traffic - the large majority - was untouched. Accuracy
is the wrong metric for a security model; detection rate is the right one.

And this is a LOWER BOUND on the danger: real attacks (FGSM, PGD) choose
the WORST-case direction rather than a random one, and do more damage at
the same perturbation size.


In [2]:
# Why: attackers can corrupt a model through its TRAINING data - and a simple
# cryptographic fingerprint makes any tampering detectable.

# Step 2: Data poisoning + integrity checking

import hashlib

print("="*80)
print("☠️  DATA POISONING TEST (real intrusion-detection training set)")
print("="*80)
print("\nAn attacker who can influence your labelled training data - a compromised")
print("labelling pipeline, a poisoned public feed - flips labels. We retrain at")
print("four contamination levels and measure the damage:\n")

rng_poison = np.random.default_rng(1)
poison_results = {}
print(f"{'Flipped':>9}{'Accuracy':>11}{'Attacks caught':>17}{'Detection lost':>16}")
for frac in [0.0, 0.05, 0.10, 0.25]:
    y_poisoned = y_train.copy()
    n_flip = int(frac * len(y_poisoned))
    idx = rng_poison.choice(len(y_poisoned), n_flip, replace=False)
    y_poisoned[idx] = 1 - y_poisoned[idx]
    m = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    m.fit(X_train, y_poisoned)
    p = m.predict(X_test)
    poison_results[frac] = {'accuracy': accuracy_score(y_test, p),
                            'recall': recall_score(y_test, p)}
    lost = poison_results[0.0]['recall'] - poison_results[frac]['recall']
    print(f"{frac:>8.0%}{poison_results[frac]['accuracy']:>11.4f}"
          f"{poison_results[frac]['recall']:>17.4f}{lost:>16.4f}")

drop25 = poison_results[0.0]['recall'] - poison_results[0.25]['recall']
drop05 = poison_results[0.0]['recall'] - poison_results[0.05]['recall']
print(f"\nIn this run, flipping 5% of labels cost {drop05:.4f} detection and")
print(f"flipping 25% cost {drop25:.4f}. Random poisoning degrades the model roughly")
print("in proportion to how much of the data it corrupts - the forest averages")
print("over many trees, which absorbs a lot of random noise.")
print("\nCompare that with the evasion attack in Step 1, which destroyed detection")
print("with a 1% nudge at inference time. Random poisoning is the WEAK attack.")
print("TARGETED poisoning (backdoors) is the scary one: it plants a specific")
print("trigger with a tiny number of samples while overall metrics still look fine,")
print("so none of the numbers in this table would warn you.")

print("\n" + "="*80)
print("🔏 INTEGRITY CHECK: DETECT TAMPERED TRAINING DATA")
print("="*80)
# Hash the exact bytes of X and y: any single changed label produces a
# completely different fingerprint (see the SHA-256 demo in Notebook 06).
def dataset_fingerprint(X, y):
    h = hashlib.sha256()
    h.update(np.ascontiguousarray(X).tobytes())
    h.update(np.ascontiguousarray(y).tobytes())
    return h.hexdigest()[:16]

fp_original = dataset_fingerprint(X_train, y_train)
print(f"\nFingerprint at data-collection time: {fp_original}")

y_tampered = y_train.copy(); y_tampered[0] = 1 - y_tampered[0]  # 1 label changed
fp_now = dataset_fingerprint(X_train, y_tampered)
print(f"Fingerprint before training:         {fp_now}")
print("MATCH -> safe to train" if fp_now == fp_original
      else "MISMATCH -> data was modified since collection: STOP and investigate")
print(f"\nExactly ONE of {len(y_train):,} labels was altered, and the fingerprint")
print("is unrecognisable. That is the property you want: tampering cannot be subtle.")

print("""
Secure AI development practices this demonstrates:
  1. Test robustness to input perturbation BEFORE deployment - and test the
     metric that matters (detection rate), not the one that looks good (accuracy)
  2. Model the ATTACKER: they perturb their own inputs, not the whole test set
  3. Track provenance + validate labels to resist poisoning
  4. Hash datasets (and models!) so tampering is detectable
  5. Combine with Unit-3 privacy controls: encryption, DP, access control
""")

☠️  DATA POISONING TEST (real intrusion-detection training set)

An attacker who can influence your labelled training data - a compromised
labelling pipeline, a poisoned public feed - flips labels. We retrain at
four contamination levels and measure the damage:

  Flipped   Accuracy   Attacks caught  Detection lost
      0%     0.9994           0.9989          0.0000


      5%     0.9908           0.9889          0.0100


     10%     0.9721           0.9672          0.0317


     25%     0.8737           0.8772          0.1217

In this run, flipping 5% of labels cost 0.0100 detection and
flipping 25% cost 0.1217. Random poisoning degrades the model roughly
in proportion to how much of the data it corrupts - the forest averages
over many trees, which absorbs a lot of random noise.

Compare that with the evasion attack in Step 1, which destroyed detection
with a 1% nudge at inference time. Random poisoning is the WEAK attack.
TARGETED poisoning (backdoors) is the scary one: it plants a specific
trigger with a tiny number of samples while overall metrics still look fine,
so none of the numbers in this table would warn you.

🔏 INTEGRITY CHECK: DETECT TAMPERED TRAINING DATA

Fingerprint at data-collection time: 5db3ed4462cafd93
Fingerprint before training:         44b34eeee7ff384c
MISMATCH -> data was modified since collection: STOP and investigate

Exactly ONE of 18,200 labels was altered, and the fingerprint
is unrecognisable. That is the property you want: t

---

## ➡️ Transition to Unit 4: Transparency and Accountability

### What We've Accomplished

We've completed the five core examples of Unit 3: Privacy and Security! (Notebooks 06 and 07 offer extra hands-on practice with encryption and anonymization - do them before starting Unit 4.) We've learned:
- ✅ How to protect data (encryption, anonymization)
- ✅ How to use advanced privacy technologies (homomorphic encryption, SMPC)
- ✅ How to use differential privacy (mathematical guarantees)
- ✅ How to ensure GDPR compliance (regulatory requirements)
- ✅ How to build secure AI systems (security practices)

### The Next Challenge: Transparency and Accountability

**Privacy and security are important, but they're not the only ethical concerns!**

As we build AI systems, we also need to consider:
- **Transparency**: How do we explain AI decisions?
- **Accountability**: Who is responsible for AI outcomes?
- **Explainability**: How do we make AI understandable?
- **Auditability**: How do we track and verify AI behavior?

**The Problem**: We've learned about privacy and security, but **AI systems also raise transparency and accountability concerns**:
- AI systems make decisions that affect people
- AI systems may be "black boxes" that are hard to understand
- AI systems need to be explainable and auditable
- AI systems need clear accountability mechanisms

**This is exactly what we'll learn in Unit 4: Transparency and Accountability!**

---

## ➡️ Next Steps

**You've completed Unit 3!** Now you understand:
- ✅ How to protect data and ensure privacy
- ✅ How to comply with regulations
- ✅ How to build secure AI systems

**Next Unit**: `unit4-transparency-accountability/`
- Learn about explainable AI (XAI)
- Understand accountability frameworks
- Master transparency requirements
- Build explainable and accountable AI systems

**Congratulations!** 🎉 You've completed Unit 3 and learned how to build private, secure, and compliant AI systems!

## 📚 References

1. Goodfellow, I. J., Shlens, J. & Szegedy, C. (2015). *Explaining and Harnessing Adversarial Examples*. ICLR 2015. <https://arxiv.org/abs/1412.6572>
2. Papernot, N., McDaniel, P., Sinha, A. & Wellman, M. (2016). *Towards the Science of Security and Privacy in Machine Learning*. <https://arxiv.org/abs/1611.03814>
3. NIST (2023). *Artificial Intelligence Risk Management Framework (AI RMF 1.0)*. NIST AI 100-1. <https://www.nist.gov/itl/ai-risk-management-framework>